# Module 2: Working with DAX
### Week 10 | June 9, 2026

Module 1 was the *before-the-model* language (M / Power Query, cleaning and shaping). Module 2 is the *after-the-model* language: **DAX** (Data Analysis Expressions), the formula language for analysis that recalculates based on what the report is showing. This builds directly on the Week 9 DAX foundation, now going deeper on syntax, data types, and evaluation context.

This module covers three sections: 2.A Syntax Fundamentals (this notebook), 2.B Calculated Columns, and 2.C Measures. The class also threaded four *practical* topics through these: Measures, Calculated Columns, Calculated Tables, and Row-Level Security.

## By End of Module 2 (2.A portion) I Should Be Able To:
- [ ] Name the four types of DAX calculations and what each is for
- [ ] Write a valid DAX formula (starts with `=`, returns a scalar)
- [ ] Explain row context vs filter context at a high level
- [ ] Use fully qualified names (`'Table'[Column]`) correctly
- [ ] Apply operator precedence and predict evaluation order
- [ ] Identify DAX scalar data types and the table data type
- [ ] Read a nested function call from the inside out

<h2 style="color:steelblue">Section 2.A: DAX Syntax Fundamentals</h2>

### What is DAX?

Data Analysis Expressions (**DAX**) is a formula expression language used in Analysis Services, Power BI, and Power Pivot in Excel. DAX formulas include **functions, operators, and values** to perform advanced calculations and queries on data in related tables and columns in tabular data models.

The four practical topics this module threads through:

`Measures` | `Calculated Columns` | `Calculated Tables` | `Row-Level Security`

> 💡 Week 9 anchor: DAX runs *after* data is in the model and recalculates based on report filter context — unlike Power Query/M, which runs *before* and produces fixed values.

### Python vs DAX

Where DAX sits relative to the Python I know:

| Comparison Area | Python | DAX |
|---|---|---|
| **Primary Purpose** | General-purpose: data analysis, automation, ML, web/app dev | Formula language in Power BI, Excel Power Pivot, Analysis Services for data modeling + analytics |
| **Where It Runs** | Jupyter, VS Code, PyCharm, Python scripts | Inside Power BI, Power Pivot, SQL Server Analysis Services data models |
| **Data Processing** | Constructs: variables, loops, functions, classes; libraries like pandas, NumPy | Formulas that operate on tables, columns, rows, and filter contexts within a data model |
| **Typical Use Cases** | Cleaning, stats, ML, automation, predictive analytics, custom apps | Measures, calculated columns, calculated tables, KPIs, BI calculations |
| **Learning Curve** | Programming concepts: syntax, variables, functions, loops, OOP | Easier if you know Excel formulas, but requires row context, filter context, and relationships |

> 💡 The honest takeaway: DAX is *not* a general programming language. It's a focused formula language. The hard part isn't syntax (it's Excel-like), it's the **context** concepts, which is exactly what 2.A starts building.

### 2.A.1 Calculations

DAX adds values to your existing data model. There are **four types** of calculations:

| Calculation Type | Adds Value At | Key Trait |
|---|---|---|
| **Calculated Columns** | Row level | Calculated row by row as soon as added |
| **Measures** | Summary level | Requires context to be calculated |
| **Calculated Tables** | Model level | New tables derived from other model data |
| **Row Filters** | Row level (security) | Used for row-level security (RLS) |

Every calculation is expressed through a **formula**.

> 💡 These are the same four the class flagged on the "What is DAX?" slide. Note the split: columns/measures add *values*, calculated tables add *tables*, row filters add *security*.

### 2.A.2 Formulas

A DAX formula **always starts with an equal sign `=`**.

- There may be a **name** on the left of the `=`, storing the result so it can be used as a field in a report.
- A DAX formula **cannot modify** existing values in the model — it only **adds** new values.
- After the `=`, you provide an expression that evaluates to a **scalar** (a single value), or that can be converted to one. This includes:
  - Operators, constants, and values
  - References to columns or tables (DAX always uses tables/columns as function inputs — never an arbitrary array of values)
  - The result of a function and its arguments (some functions return a *table*, which must be wrapped in a function that turns it into a scalar)

> 💡 "Cannot modify, only add" is a real contrast with pandas, where `df["col"] = ...` overwrites freely. DAX is additive.

### 2.A.3 Formula Evaluation — Row Context vs Filter Context

Formulas behave **differently** depending on whether they're in a calculated column or a measure, because they evaluate within an **evaluation context**:

| Context | Most relevant for | What it does |
|---|---|---|
| **Row Context** | Calculated columns + inside an iterator | Lets an expression reference a column value *outside* an aggregate (i.e. "the current row") |
| **Filter Context** | Measures | Implicit + explicit filters limit the data the formula sees |

This is *the* core concept of the whole module. ⭐

> 💡 Week 9 connection: this is the formal name for why a measure recalculates per slicer (filter context) but a calculated column is fixed at load (row context). pandas parallel: row context ≈ operating inside `df.apply(axis=1)` where you see one row; filter context ≈ the `df[df["region"]=="East"]` slice that's active when a `.sum()` runs.

### 2.A.4 Formula Examples

| Formula | Result |
|---|---|
| `= 8` | The number 8 |
| `= "Sales"` | The string "Sales" |
| `= 'Sales'[Amount]` | The `[Amount]` value for the current **row context** (used in the Sales table) |
| `= ( 0.03 * [Amount] )` | 3% of `[Amount]` in the current row context |
| `= 0.03 * [Amount]` | Same math, but won't *display* as a percentage unless you format the column |
| `= PI()` | The constant π |

> ⚠️ That last pair matters: the formula computes the value, but **formatting** (how it displays as a %) is a separate setting in the table. Don't confuse "is a percentage" with "displays as a percentage."

### 2.A.5 Naming Requirements

- Each **table** must have a unique name (just like SQL databases).
- Within a table, each **column** must have a unique name.
- Object names are **case-insensitive** → `SALES` is the same as `Sales`. ⚠️
- Every column or measure must belong to a **specific table**.

> ⚠️ Big contrast with M! M (Module 1) was **case-sensitive** (`Source` ≠ `source`). DAX is **case-insensitive** (`SALES` = `Sales`). Two languages, opposite rules — easy to mix up.

### 2.A.6 Fully Qualified Names

When you use a table or column as a function input, you must **qualify** the name.

- **Table names** with spaces, reserved keywords, or special characters go in **single quotes**: `'European Sales'`. Not required for simple one-word names.
- **Column names** go in **square brackets**: `[Amount]`. This is **always** required.
- A **fully qualified column name** includes both: `'Online Sales'[Sales Amount]`.

A fully qualified name is **required** when referencing a column as an argument to: `VALUES()`, `ALL()` / `ALLEXCEPT()`, `RELATEDTABLE()`, any **time intelligence** function, or in a filter argument for `CALCULATE()` / `CALCULATETABLE()`.

> 💡 Reminder so it doesn't blur with M: in DAX, `[ ]` = a column, `' '` = a table name with spaces. (In M, `[ ]` was a record lookup and `{ }` was a list index. Different language, different brackets.)

### 2.A.7 Naming Examples

| Object Type | Example |
|---|---|
| Table name | `Sales` |
| Table name (with space) | `'European Sales'` |
| Fully qualified column name | `Sales[Amount]` |
| Fully qualified measure name | `Sales[YTD Profit]` |
| Unqualified column name | `[Amount]` |
| Fully qualified column name | `'Canada Sales'[Qty]` |

### 2.A.8 Operators

Operators by category:

| Category | Operators |
|---|---|
| **Arithmetic** | `+`  `-`  `*`  `/`  `^` |
| **Comparison** | `=`  `==`  `>`  `<`  `>=`  `<=`  `<>` |
| **Text Concatenation** | `&` |
| **Logical** | `&&` (and)  `||` (or)  `IN` |

Many operators get combined in a single formula.

> 💡 Mostly Python-like, with twists: text join is `&` (not `+`), logical and/or are `&&` / `||` (not `and`/`or`), and `<>` means "not equal." `IN` works like Python's `in` for set membership.

### 2.A.9 Operator Precedence

Most formulas have multiple calculations, and the order affects the result. Evaluation goes **left to right**, following this precedence:

1. Parentheses `( )`
2. Exponent `^`
3. Multiplication / Division `*` `/`
4. NOT `!`
5. Addition / Subtraction `+` `-`
6. Concatenation `&`
7. Comparison `=` `==` `<` `>` `<=` `>=` `<>`

As in SQL and Python, **use parentheses** to control the order explicitly.

> 💡 Same instinct as everywhere else: when unsure, parenthesize. Don't trust yourself to remember the table under pressure.

### 2.A.10 Operator Precedence Examples

Computing one operator per step:

    = 3 + 3 / 1 * 4 - 2
    = 3 + 3 * 4 - 2
    = 3 + 12 - 2
    = 15 - 2
    = 13

Same expression, parentheses change the order and the answer:

    = (3 + 3) / 1 * (4 - 2)
    = 6 / 1 * (4 - 2)
    = 6 / 1 * 2
    = 6 * 2
    = 12

⚠️ The minus sign is an operator and participates in precedence — this differs from algebra:

    = -2^2      // results in -4
    = (-2)^2    // results in 4

> 💡 That last one bites people. `-2^2` applies `^` before the negation, giving `-4`. Parenthesize when you mean "negative two, squared."

### 2.A.11–2.A.12 Scalar Data Types

> Merged: this cell covers two guide subsections — 2.A.11 (Scalar Data Types) and 2.A.12 (Scalar Data Types, continued).

Every value has a **type** (set by how the source column was defined). A calculated column or measure formula must return a **scalar** (single value). The scalar types:

| Type | Notes |
|---|---|
| **Integer** | Whole numbers, positive or negative (range ±2^63) |
| **Real Number** | Decimals; ~15 significant digits |
| **Boolean** | True / False |
| **String** | Alphanumeric unicode text |
| **Date/Time** | Stored as a floating-point: integer part = days since Dec 30 1899, decimal = time. Add/subtract 1 to move by a day |
| **Currency** | Fixed precision, 4 decimal digits |
| **Blank** | DAX's version of a SQL null. Create with `BLANK()`, test with `ISBLANK()` |

> 💡 `Blank` is the one to remember: it's DAX's `null`/`NaN`. The Date/Time-as-a-float detail explains why you can do `date + 1` to get tomorrow.

### 2.A.13 The Table Data Type

Beyond scalars, DAX has a **table data type**.

- Many functions **require** a table reference as input.
- Other functions **return** a table, which can feed into another function.
- Some functions accept an **expression that evaluates to a table**; others need a reference to a **base table** (an evaluated table won't work).
- A **calculated table** formula must return a **table value** (not a scalar).

> 💡 This is the "functions that return tables must be wrapped" point from 2.A.2, made concrete. It's also why `SUMMARIZE` (the calculated-table example from class) returns a whole table, not one number.

### 2.A.14 Data Type Conversions (Implicit)

Each function has input/output type requirements. DAX will **automatically convert** types when it can — called **implicit conversion**:

| Expression | Returns | Why |
|---|---|---|
| `= "7/4/2019"` | A date | DAX parses the string into a date |
| `= "July 4th 2019"` | Same date | Same parsing |
| `= TRUE + 2` | `3` | TRUE implicitly becomes `1` |
| `= "12" + 12` | `24` | Text `"12"` converts to number |

If implicit conversion **fails**, DAX returns an **error**.

> 💡 Python is stricter — `"12" + 12` throws a TypeError. DAX quietly coerces. Convenient, but watch for surprises when a string sneaks into math.

### 2.A.15 Implicit Conversions with Operators

Each operator has its own rules for mixing data types. Most are intuitive, some aren't:

| Operation | Result | Intuitive? |
|---|---|---|
| `REAL * CURRENCY` | CURRENCY | ✅ Yes — 2.5 of a $20 item = $50 |
| `REAL + CURRENCY` | REAL | ⚠️ Less so |
| `DateTime / DateTime` | REAL | ⚠️ Unusual (use date functions instead) |

When in doubt, check the Microsoft docs — they have full rule tables per operator.

> 💡 Don't memorize these. Just know that mixing types has rules, and the docs are the source of truth.

### 2.A.16 Function Fundamentals

DAX is a **functional language** — execution flows through function calls. Each function has:

- Its **name**
- A list of **input parameters**
- A **return type** (implicit)

There are **250+ functions** across many categories. (Microsoft's DAX function reference is the canonical list.)

> 💡 "Functional language" echoes M from Module 1, but DAX's flavor is different — it's about composing function calls over tables/columns, with context doing the heavy lifting.

### 2.A.17–2.A.18 Function Syntax & Order of Evaluation

> Merged: this cell covers two guide subsections — 2.A.17 (Function Syntax) and 2.A.18 (Order of Evaluation).

**Function syntax:** first step learning any function is the Microsoft docs. You pass **arguments into input parameters inside parentheses**.

**Order of evaluation:** functions evaluate **from the inside out** (innermost first). Example:

    = SUMX(
        FILTER(
            VALUES('Date'[Year]),
            IF(
                'Date'[Year] >= 2000,
                [Sales Amount] * 100,
                [Sales Amount] * 90
            )
        )
      )

This runs innermost first: `IF` → `FILTER` (over `VALUES`) → `SUMX` wraps it all.

> 💡 "Inside out" is exactly how nested pandas/Python calls evaluate too — `sum(filter(...))` runs `filter` before `sum`. Reading DAX is reading from the deepest parenthesis outward.

### 2.A.19–2.A.20 Formatting Formulas & Using Online Reference

> Merged: this cell covers two guide subsections — 2.A.19 (Formatting Formulas) and 2.A.20 (Using Online Reference Materials).

**Formatting:** Power BI lets you expand the formula bar for multiple lines. You'll often think about formulas **outside-in** or **inside-out** rather than strictly left-to-right or top-to-bottom.

**Reference:** with 250+ functions and new ones added several times a year, a quick search to Microsoft's official DAX reference saves time. Worth browsing the directory to see what exists.

> 💡 Same habit as M's `powerqueryformatter.com` and the Microsoft M reference — format for readability, and lean on the docs rather than memorizing.

### 2.A.21–2.A.22 Common Functions by Category

> Merged: this cell covers two guide subsections — 2.A.21 and 2.A.22 (Common Functions by Category).

The starter toolkit, grouped:

| Category | Functions |
|---|---|
| **Aggregation** | `SUM()`, `AVERAGE()`, `MIN()`, `MAX()`, `SUMX()` and other X functions |
| **Counting** | `COUNT()`, `COUNTA()`, `COUNTBLANK()`, `COUNTROWS()`, `DISTINCTCOUNT()` |
| **Logical** | `AND()`, `OR()`, `NOT()`, `IF()`, `IFERROR()`, `SWITCH()` |
| **Information** | `ISBLANK()`, `ISNUMBER()`, `ISTEXT()`, `ISNONTEXT()`, `ISERROR()` |
| **Text** | `CONCATENATE()`, `REPLACE()`, `SEARCH()`, `UPPER()`, `FIXED()` |
| **Date** | `DATE()`, `HOUR()`, `NOW()`, `EOMONTH()`, `WEEKDAY()`, `DATEDIFF()` |

> 💡 The "X functions" (`SUMX`, etc.) are the iterators — they go row by row using **row context**, then aggregate. That's the `SUMX(samplesales, [Qty] * [Price])` pattern from the class measure example, and it connects straight to 2.A.3's row context.

### 🔁 Confusion Corner (2.A)

**"Row context vs filter context — what's the actual difference?"** ⭐ The whole module hinges on this. Row context = "which row am I on right now" (calculated columns, iterators). Filter context = "which slice of data is visible right now" (measures, driven by slicers/filters). A calc column has row context at load; a measure has filter context at report time.

**"M was case-sensitive, is DAX?"** ⚠️ No — DAX is case-**insensitive** (`SALES` = `Sales`). Opposite of M. Don't carry the M rule over.

**"When do I need the single quotes vs brackets?"** `'Table Name'` (single quotes, only if it has spaces) and `[Column]` (square brackets, always). Fully qualified = `'Table'[Column]`.

**"Why did `-2^2` give -4?"** The `^` binds tighter than the negative sign, so it's `-(2^2)`. Use `(-2)^2` if you want `4`. ⚠️

**"`"12" + 12` = 24?!"** DAX does implicit type conversion — it coerces the string to a number. Python would error; DAX just converts. Handy but sneaky.

**"Inside-out evaluation?"** Nested functions run innermost first. Read from the deepest parenthesis outward, like nested calls in Python.

<h2 style="color:steelblue">Section 2.B: Calculated Columns</h2>

### 2.B.1 Calculated Columns

A calculated column adds a **new field to a table** in the model. The defining traits:

- A calculated column is **computed for each row** of the table it belongs to.
- Its value can be used in **other calculations** as if it were a regular column.
- As soon as you create it in the UI, values are calculated for every row **immediately** and shown in the table.
- Values are **stored in the in-memory data model**, recalculated on refresh, lost when the model unloads, and recomputed on reload (so closing/opening Power BI Desktop triggers a recompute).

⚠️ **Limitation:** calculated columns are **not supported with DirectQuery** real-time connections, because Power BI can't know when external records change. Only data in the **current row** can be used in the expression.

> 💡 Week 9 anchor: this is the "row by row, stored at load" calculation, like `df['col'] = df['a'] * df['b']` — NOT `.agg()`. It can't respond to filter context. That's why a percentage-of-total has to be a measure, not a calculated column.

### Creating a Calculated Column — Worked Scenario

Using the `samplesales` table created earlier (from `samplesales.csv`):

| Product | Quantity | Unit Price |
|---|---|---|
| Laptop | 2 | 800 |
| Monitor | 3 | 250 |
| Mouse | 5 | 20 |
| Keyboard | 4 | 50 |

Steps:

1. In the **Fields pane**, right-click the `samplesales` table.
2. Select **New Column**.
3. Type the DAX formula in the **Formula Bar**.

Formula:

    Revenue = samplesales[Quantity] * samplesales[Unit Price]

This **creates a new column stored in the data model**, computed per row → Laptop = 1600, Monitor = 750, Mouse = 100, Keyboard = 200.

---

The class also showed the same idea on a `Sales` table:

| OrderID | Product | Quantity | UnitPrice |
|---|---|---|---|
| 1001 | Laptop | 2 | 800 |
| 1002 | Mouse | 5 | 25 |
| 1003 | Monitor | 1 | 300 |

    Line Total = Sales[Quantity] * Sales[UnitPrice]

→ 1600, 125, 300 per row.

> ⚠️ Contrast with a measure: New **Column** (stored, per row) vs New **Measure** (dynamic, per filter context). Same right-click menu, very different behavior.
> 💡 Both are pure per-row math — `df['revenue'] = df['qty'] * df['price']`. No aggregation, so they're columns, not measures.

### 2.B.2 Calculated Column Examples

Four patterns, grouped by what they produce:

| Pattern | Type | Formula |
|---|---|---|
| **Combine Text — Full Name** | Text | `[Full Name] = Customer[Last Name] & ", " & Customer[First Name]` |
| **Boolean Flag — Late Shipment** | Boolean | `[Shipped Late] = [ShipDate] > [RequiredDate]` |
| **Label Assembly — Quarter Name** | Text | `[Quarter Name] = [Year] & " Q" & [Quarter]` |
| **Financial Calculation — Net Sale** | Currency | `[Net Sale] = [Price] * [Quantity] * ( 1 - [DiscountPercent] )` |

What each shows:
- **Full Name** uses `&` to concatenate (DAX's text join, not `+`).
- **Shipped Late** produces a TRUE/FALSE flag you can filter or use in conditional formatting.
- **Quarter Name** builds a display label like "2024 Q3" for an axis or slicer.
- **Net Sale** computes line-level net revenue (price × quantity, discount-adjusted) — all within the **current row context**.

> 💡 Every one of these works row by row. `[Net Sale]` is exactly `df['net'] = df['price'] * df['qty'] * (1 - df['discount'])` in pandas — no aggregation, just per-row math.

### 2.B.3 Row Context

When a calculated column is evaluated, it runs **once for each row**, within the context of *that* row. This is called **row context**.

⚠️ Row context only sees the **current row**. To pull values from **outside** the current row (i.e. from a related table), you need a special function to **walk the relationships** of the model:

- `RELATED()`
- `RELATEDTABLE()`
- `USERELATIONSHIP()`

> 💡 This is the formal version of 2.A.3's row context, now made concrete. pandas parallel: a calculated column is like `df.apply(..., axis=1)` where each call sees one row — and `RELATED()` is the `merge` you'd need to bring in a column from another table.

### 2.B.4 RELATED()

`RELATED()` navigates a **predefined active relationship** to grab columns from **another table**.

- It only works where a relationship returns a **single row** — so the row-context table must be on the **many side** of a one-to-many (or a one-to-one).
- Common use: **flatten normalized data**. Say the model has `Product`, `ProductSubcategory`, and `ProductCategory`, and you want the names folded into `Product`:

    [Subcategory Name] = RELATED(ProductSubcategory[SubcategoryName])

    [Category Name] = RELATED(ProductCategory[CategoryName])

> 💡 `RELATED()` is a **lookup across a relationship** — like a `df.merge(other, on=key)` that pulls one column from the related table into the current row. This is the Week 9 "merge Category into Product" decision, now done in DAX instead of Power Query.

### 2.B.5 RELATEDTABLE()

`RELATEDTABLE()` navigates a relationship to grab **multiple rows** from a related table — it **returns a table**, not a scalar.

⚠️ Because a calculated column must return a **scalar**, the table from `RELATEDTABLE()` has to be collapsed into a single value first. **Iterator functions** (`SUMX`, `COUNTX`, `MINX`, `MAXX`) do that.

Example — total sales per current row, summing a related table's line totals:

    TotalSales = SUMX(
        RELATEDTABLE(SalesOrderDetail),
        SalesOrderDetail[LineTotal]
    )

`SUMX` iterates every related `SalesOrderDetail` row (only those tied to the current row context) and sums `[LineTotal]`.

> 💡 `RELATED()` = grab one value from the *one* side. `RELATEDTABLE()` = grab many rows from the *many* side, then aggregate with an X-function. Like `df.merge` (one value) vs `groupby().agg()` on the related rows (many → one).

### 2.B.6 USERELATIONSHIP()

`USERELATIONSHIP()` is used **inside `CALCULATE()`** to temporarily activate an **inactive relationship** just for that calculation.

- Use it when a calculation needs to follow a **different relationship** than the active default (e.g. aggregate sales by **ship date** instead of **order date**).
- Why it's needed: xVelocity models allow only **one active relationship** between two tables. Extra relationships get disabled to avoid ambiguity.
- This comes up constantly with **Date tables**, where one order might have Order Date, Arrive By Date, and Ship Date all pointing at the same Date table.

An alternative is **role-playing tables** (covered later).

> 💡 Think of it as "for this one calculation, use the *other* relationship." Only one relationship can be active at a time, so `USERELATIONSHIP()` flips to an inactive one temporarily.

### 🔁 Confusion Corner (2.B)

**"Calculated column vs measure — when which?"** ⭐ Column = stored, row-by-row, fixed at load, can't react to filters. Measure = dynamic, recalculates per filter context. If it needs to respond to a slicer (like % of total), it's a measure. If it's a per-row attribute (like Full Name), it's a column.

**"`RELATED()` vs `RELATEDTABLE()`?"** `RELATED()` pulls **one value** from the *one* side of a one-to-many (returns a scalar directly). `RELATEDTABLE()` pulls **many rows** from the *many* side (returns a table) — so you must wrap it in an iterator like `SUMX` to get a scalar. ⚠️

**"Why does `RELATEDTABLE()` need `SUMX`?"** Because a calculated column must return a single value, and `RELATEDTABLE()` hands back a whole table. The X-function collapses the table to one number.

**"What's `USERELATIONSHIP()` for again?"** Only one relationship between two tables can be active. `USERELATIONSHIP()` (inside `CALCULATE()`) temporarily switches to an inactive one — classic for Date tables with multiple date columns.

**"Why no calculated columns with DirectQuery?"** DirectQuery is live external data, so Power BI can't tell when source rows change to trigger a recompute. Only current-row data is usable.

<h2 style="color:steelblue">Section 2.C: Measures</h2>

### 2.C.1 Measures

Measures are the **second of four DAX calculation types** — and arguably **the most important feature of DAX**.

- They provide **summaries** that concisely tell you what all the detail beneath them means.
- This kind of calculation **can't be done in Power Query or SQL** without severe performance hits.
- Measures are **dynamic**: results change depending on context (a Power BI report, Excel PivotTable, or PivotChart).
- Added at the **table level**, but they do **NOT** provide one value per row.
- Computed in **real-time when a report queries the model** — not when you create them, only when they're called in a report.
- Created using the **DAX formula bar** in the model designer.

> 💡 Week 9 anchor: a measure is the "saved calculation that recalculates per filter context" — like `df.groupby().sum()` but **reactive** to whatever the report is filtering. A calculated column is stored per row; a measure is computed on demand for the current slice.

### Creating a Measure — Worked Scenario

Using the `samplesales` table:

| Product | Quantity | Unit Price |
|---|---|---|
| Laptop | 2 | 800 |
| Monitor | 3 | 250 |
| Mouse | 5 | 20 |
| Keyboard | 4 | 50 |

Steps:

1. Load `samplesales.csv` in Power BI Desktop.
2. Go to **Report** or **Data View**.
3. In the **Fields pane**, right-click the `samplesales` table.
4. Select **New Measure**.
5. Type the DAX formula in the **Formula Bar**.

Formula:

    Total Revenue = SUMX(
        samplesales,
        samplesales[Quantity] * samplesales[Unit Price]
    )

⚠️ **A measure does NOT create a new column.** It calculates the result **dynamically** based on filters and slicers in the report.

The class also showed it on a `Sales` table:

    Total Revenue = SUMX(
        Sales,
        Sales[Quantity] * Sales[UnitPrice]
    )

> 💡 `SUMX` is an **iterator**: it goes row by row (using row context) multiplying Quantity × Price, then sums the results. Like `(df['qty'] * df['price']).sum()` — but the rows it sums depend on the report's current filter context.

### 2.C.2 Implicit Measures

Any time you add a column to a visual and what you see is an **aggregate of many values** (instead of a single value from a single row), you're using an **implicit measure**.

Example: drag `Sale Amount` into a visualization, and implicitly the formula is:

    SUM(Sales[Sale Amount])

You didn't write any DAX — Power BI auto-created the aggregation. **SUM is the default** for a numeric column.

> 💡 Quiz reinforcement: implicit measure = auto-created when a column is aggregated in a visual (not written with `CALCULATE`). And the default aggregation Power BI picks is `SUM`, so `Sale Amount` in a visual → `SUM(Sales[Sale Amount])`.

### 2.C.3 Filter Context

How data gets summarized is governed by **filter context**.

Picture a grid (matrix visual) full of numbers where **every cell is the same expression** — `SUM(Sales[Sales Amount])`. Yet each cell shows a **different number**, because the **filter context differs** at each location, so a different subset of rows is available to the calculation.

> 💡 This is *the* measure concept. Same formula, different filtered slice → different result. pandas parallel: it's as if every cell silently ran `df[<that cell's filters>]["Sales Amount"].sum()`. The filters define the slice; the measure sums whatever's left.### 2.C.3 Filter Context

How data gets summarized is governed by **filter context**.

Picture a grid (matrix visual) full of numbers where **every cell is the same expression** — `SUM(Sales[Sales Amount])`. Yet each cell shows a **different number**, because the **filter context differs** at each location, so a different subset of rows is available to the calculation.

> 💡 This is *the* measure concept. Same formula, different filtered slice → different result. pandas parallel: it's as if every cell silently ran `df[<that cell's filters>]["Sales Amount"].sum()`. The filters define the slice; the measure sums whatever's left.

### 2.C.4 Interactive Filter Context

When a user **interacts** with report items — clicking a bar, selecting a slicer, choosing a region — that selection **changes the filter context**.

All other report items then **limit themselves** to display data based on this new filter context.

> 💡 Quiz reinforcement: this is *why the same measure returns different values in different parts of a report* — and why it updates live as you click. The user's selection rewrites the filter context, and every measure recalculates against it.

### 2.C.5 Example Measures

A progression from simple to compound:

**Total quantity sold** — a basic sum:

    [Total quantity sold] = SUM(Sales[Quantity])

(You *could* get this implicitly by dropping `Sales[Quantity]` in a report, but defining it explicitly lets you give it a **better name**.)

**Actual sales values** — iterate price × quantity for the current filter context:

    [Actual sales values] = SUMX(Sales, [Price] * [Quantity])

**List price values** — same idea, but pull list price from the related Product table:

    [List price values] = SUMX(Sales, RELATED('Product'[ListPrice]) * [Quantity])

**Pct of full value** — what fraction of list price the customer actually paid:

    [Pct of full value] = DIVIDE(
        SUMX(Sales, [Price] * [Quantity]),
        SUMX(Sales, RELATED('Product'[ListPrice]) * [Quantity])
    )

> 💡 Notice `RELATED()` showing up inside a measure (from 2.B) to reach the Product table, and `DIVIDE()` instead of `/` — `DIVIDE` safely handles divide-by-zero, returning blank instead of an error. This is also why "% of total" *must* be a measure: it needs filter context, which a calculated column can't see.

### 2.C.6 Measures vs Calculated Columns

The decision rules, straight from the guide:

| Use a **Calculated Column** when... | Use a **Measure** when... |
|---|---|
| You'll later want to **filter** on the resulting value | You want **percentages** in a report |
| You need the value to **support another calculated column** | You want **ratios** in a report |
| | You need **complex aggregations** |
| | You're describing how to **present a value that looks different in different filter contexts** |

> 💡 Quick gut check: if the answer must **react to slicers**, it's a measure. If it's a **fixed per-row attribute** you might filter or slice *by*, it's a column. % of total → measure. Full Name → column.

### Recap — Variables (VAR)

**Variables (`VAR`)** store intermediate calculations inside a formula — used most often in measures:

- **Store intermediate calculations** so you don't repeat them.
- **Improve readability and performance** (DAX computes the variable once).

Best practices:
- Use variables for **complex formulas**.
- Keep measures **readable and modular**.
- **Avoid repeating** calculations.
- Use **proper naming conventions**.

Pattern (conceptual):

    Measure =
    VAR Sales = SUMX(...)
    VAR Cost = SUMX(...)
    RETURN Sales - Cost

> 💡 `VAR` is just naming a value before using it — like `total = a * b` in Python, then reusing `total` — instead of writing the same expression twice. The `Pct of full value` measure above would be cleaner with two VARs for the numerator and denominator.

### 🔁 Confusion Corner (2.C)

**"Implicit vs explicit measure?"** Implicit = auto-created when you drop a numeric column in a visual (default `SUM`). Explicit = one you write yourself in the formula bar (and can name properly). Q4/Q5 tested this.

**"Row context vs filter context, one more time?"** ⭐ Row context = "which row" (calculated columns, iterators). Filter context = "which filtered slice" (measures). A measure summarizes based on **filter context** — that's Q6.

**"Why does the same measure show different numbers?"** Because each cell/visual/slicer applies a different **filter context**, so the measure sums a different subset of rows. Same formula, different slice — that's Q7.

**"Why `DIVIDE()` instead of `/`?"** `DIVIDE()` handles divide-by-zero safely (returns blank, not an error). Use it for any ratio or percentage measure.

**"When column, when measure?"** Column if you'll filter *on* it or feed another column. Measure for %, ratios, complex aggregations, or anything that should change with filter context.

**"Does a measure cost storage like a column?"** No — a column is stored in memory at load; a measure stores only the formula and runs on demand. That's why measures don't bloat the model.

### ⭐ When Would I Use This? 

**Calculated Column** (per-row, stored):
- A `Full Name`, a `Line Total`, a "shipped late" flag — anything I'll later filter or slice *by*.

**Measure** (reactive to filters):
- `Total Revenue`, % of total, ratios, KPIs — anything that should change as the user filters.

**Relationship functions:**
- `RELATED` → pull a value from a related table (flatten Category into Product).
- `RELATEDTABLE` + `SUMX` → aggregate related rows per current row.
- `USERELATIONSHIP` → sum by ship date instead of order date when a Date table has multiple date columns.

**Calculated Table:**
- `SUMMARIZE` → a "total quantity by product" grouped table to use elsewhere.

**Row-Level Security:**
- East-region users only see East rows, via a DAX role filter.

**Handy helpers:**
- `DIVIDE` → any ratio or percentage measure, to dodge divide-by-zero.
- `VAR` → name an intermediate value instead of repeating it.

### Key Takeaways🎯

Module 2 is the *after-the-model* language: **DAX**. M cleaned and shaped data before it landed; DAX does the analysis once it's in. And the whole thing rises or falls on one word — **context**.

**The four calculations, and when to grab each:**
- **Calculated column** → per-row, stored at load, frozen. Good for attributes I'll filter *by*. Can't react to slicers.
- **Measure** → stores only a formula, recomputes live for whatever slice the report shows. The reactive one.
- **Calculated table** → returns a whole new table (`SUMMARIZE` ≈ a `groupby`).
- **Row filters** → the security layer (RLS), deciding which rows a user can even see.

**The big idea — row context vs filter context:**
Row context = "which row am I on" (columns, iterators like `SUMX`). Filter context = "which filtered slice is active" (measures). That's why the same `SUM(Sales[Amount])` shows a different number in every cell of a matrix — same formula, different slice. Click a slicer, the context changes, everything recalculates. That's the whole point of a measure, not a bug.

**Syntax things that save grief:**
Case-*insensitive* (opposite of M). Columns are `[Brackets]`, tables with spaces are `'Quotes'`. Text joins with `&` not `+`. And `-2^2` = `-4`, so parenthesize when unsure.

**The relationship family:**
- `RELATED` → one value from the *one* side.
- `RELATEDTABLE` → many rows from the *many* side (wrap in `SUMX`/`COUNTX` for a scalar).
- `USERELATIONSHIP` → temporarily flip to an inactive relationship inside `CALCULATE`.
- Plus `DIVIDE` for safe ratios, and `VAR` to name values instead of repeating them.

**Bottom line:** reacts to filters → measure. Fixed per-row thing → column. New derived table → calculated table. Who sees what → RLS. Nail the context model and the rest is just looking up function names. ✅